<a href="https://colab.research.google.com/github/haniyathul/TESIS-HANIY/blob/main/SIMULASI%20PEMBENTUKAN%20ESF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# FILE 2 — EIGENVECTOR SPATIAL FILTERING (ESF)
# ============================================================
# TUJUAN:
# 1. Membaca DATA_GRID10 ... DATA_GRID50 dari repository
# 2. Memanggil / membentuk W yang sama dengan File 1
# 3. Membuat W simetris KHUSUS untuk ESF
# 4. Membentuk Moran operator: M W M
# 5. Full eigendecomposition
# 6. Menyimpan SEMUA n-1 Moran eigenvectors
# 7. BELUM ADA SELEKSI
# ============================================================

import os
import gc
import shutil

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.sparse import (
    csr_matrix,
    load_npz,
    save_npz
)
from scipy.linalg import eigh


# ============================================================
# PATH REPOSITORY
# ============================================================

REPO_DIR = "/content/TESIS-HANIY"

DATA_DIR = os.path.join(
    REPO_DIR,
    "data"
)

WEIGHT_DIR = os.path.join(
    REPO_DIR,
    "weights"
)

ESF_DIR = os.path.join(
    REPO_DIR,
    "esf"
)


os.makedirs(
    WEIGHT_DIR,
    exist_ok=True
)

os.makedirs(
    ESF_DIR,
    exist_ok=True
)


# ============================================================
# CEK REPOSITORY
# ============================================================

if not os.path.exists(REPO_DIR):

    raise FileNotFoundError(
        "Repository /content/TESIS-HANIY tidak ditemukan."
    )


if not os.path.exists(
    os.path.join(
        REPO_DIR,
        ".git"
    )
):

    raise RuntimeError(
        "/content/TESIS-HANIY ada, tetapi bukan repository Git."
    )


# ============================================================
# CEK DATA FILE 1
# ============================================================

EXPECTED_DATA = [

    "DATA_GRID10.csv",
    "DATA_GRID20.csv",
    "DATA_GRID30.csv",
    "DATA_GRID40.csv",
    "DATA_GRID50.csv"

]


missing_data = [

    file_name

    for file_name in EXPECTED_DATA

    if not os.path.exists(
        os.path.join(
            DATA_DIR,
            file_name
        )
    )

]


if len(missing_data) > 0:

    raise FileNotFoundError(
        f"Data belum lengkap: {missing_data}"
    )


print("✅ Seluruh DATA_GRID10–50 ditemukan di repository.")


# ============================================================
# KONFIGURASI W
# ============================================================

W_CONFIG = {

    "KNN5": {
        "label": "KNN-5",
        "filename": lambda g:
            f"W_KNN5_GRID{g}.npz"
    },

    "DISTANCE": {
        "label": "Distance Band",
        "filename": lambda g:
            f"W_DISTANCE_GRID{g}.npz"
    },

    "IDW": {
        "label": "IDW",
        "filename": lambda g:
            f"W_IDW_GRID{g}.npz"
    }

}


# ============================================================
# ROW STANDARDIZATION
# ============================================================

def row_standardize_sparse(W):

    row_sum = np.asarray(
        W.sum(axis=1)
    ).ravel()


    if np.any(row_sum == 0):

        raise ValueError(
            "Ditemukan baris W tanpa tetangga."
        )


    W_std = W.multiply(
        1.0 / row_sum[:, None]
    )


    return W_std.tocsr()


# ============================================================
# KOORDINAT GRID
# ============================================================

def make_grid_coordinates(grid_size):

    n = grid_size ** 2

    idx = np.arange(n)

    x = idx % grid_size
    y = idx // grid_size


    coords = np.column_stack(
        [
            x,
            y
        ]
    ).astype(float)


    return coords


# ============================================================
# BENTUK W
# SESUAI DESAIN FILE 1
# ============================================================

def build_weight_matrix(
    grid_size,
    method
):

    coords = make_grid_coordinates(
        grid_size
    )

    n = len(coords)

    tree = cKDTree(
        coords
    )


    rows = []
    cols = []
    values = []


    # ========================================================
    # KNN-5
    # ========================================================

    if method == "KNN5":

        distance,
        indices = tree.query(
            coords,
            k=6
        )


        for i in range(n):

            neighbors = [

                j

                for j in indices[i].tolist()

                if j != i

            ][:5]


            rows.extend(
                [i] * len(neighbors)
            )

            cols.extend(
                neighbors
            )

            values.extend(
                [1.0] * len(neighbors)
            )


    # ========================================================
    # DISTANCE BAND <= 1.5
    # ========================================================

    elif method == "DISTANCE":

        neighbor_list = (
            tree.query_ball_point(
                coords,
                r=1.5
            )
        )


        for i, neighbors in enumerate(
            neighbor_list
        ):

            neighbors = [

                j

                for j in neighbors

                if j != i

            ]


            rows.extend(
                [i] * len(neighbors)
            )

            cols.extend(
                neighbors
            )

            values.extend(
                [1.0] * len(neighbors)
            )


    # ========================================================
    # LOCAL IDW
    # RADIUS = 2.5
    # ALPHA = 1
    # ========================================================

    elif method == "IDW":

        neighbor_list = (
            tree.query_ball_point(
                coords,
                r=2.5
            )
        )


        for i, neighbors in enumerate(
            neighbor_list
        ):

            for j in neighbors:

                if j == i:
                    continue


                distance = np.linalg.norm(
                    coords[i]
                    -
                    coords[j]
                )


                rows.append(i)
                cols.append(j)

                values.append(
                    1.0 / distance
                )


    else:

        raise ValueError(
            f"W method tidak dikenal: {method}"
        )


    W = csr_matrix(

        (
            values,
            (
                rows,
                cols
            )
        ),

        shape=(
            n,
            n
        )

    )


    W = row_standardize_sparse(
        W
    )


    return W


# ============================================================
# CARI W ASLI FILE 1
# ATAU RECONSTRUCT JIKA TIDAK ADA
# ============================================================

def get_weight_matrix(
    grid_size,
    method
):

    file_name = (
        W_CONFIG[method][
            "filename"
        ](
            grid_size
        )
    )


    repo_path = os.path.join(
        WEIGHT_DIR,
        file_name
    )


    # --------------------------------------------------------
    # 1. SUDAH ADA DI REPO
    # --------------------------------------------------------

    if os.path.exists(
        repo_path
    ):

        W = load_npz(
            repo_path
        )

        source = "Repository"

        return W.tocsr(), source


    # --------------------------------------------------------
    # 2. CARI OUTPUT ASLI FILE 1 DI RUNTIME
    # --------------------------------------------------------

    possible_paths = [

        os.path.join(
            "/content",
            file_name
        ),

        os.path.join(
            "/content/TESIS-HANIY_BACKUP",
            file_name
        )

    ]


    for source_path in possible_paths:

        if os.path.exists(
            source_path
        ):

            shutil.copy2(
                source_path,
                repo_path
            )


            W = load_npz(
                repo_path
            )


            print(
                f"✅ W asli ditemukan dan disalin: "
                f"{file_name}"
            )


            return (
                W.tocsr(),
                "File 1 Original"
            )


    # --------------------------------------------------------
    # 3. RECONSTRUCT
    # --------------------------------------------------------

    print(
        f"⚠️ {file_name} tidak ditemukan. "
        f"W direkonstruksi dari desain File 1."
    )


    W = build_weight_matrix(
        grid_size,
        method
    )


    save_npz(
        repo_path,
        W
    )


    return (
        W,
        "Reconstructed"
    )


# ============================================================
# VALIDASI W
# ============================================================

def validate_weight_matrix(W):

    row_sum = np.asarray(
        W.sum(axis=1)
    ).ravel()


    row_sum_error = np.max(
        np.abs(
            row_sum - 1.0
        )
    )


    diagonal_max = np.max(
        np.abs(
            W.diagonal()
        )
    )


    return (
        row_sum_error,
        diagonal_max
    )

RuntimeError: /content/TESIS-HANIY ada, tetapi bukan repository Git.

## FUNGSI PEMBENTUKAN SEMUA ESF